<a href="https://colab.research.google.com/github/olucasaguiar/projeto-simulacao-opiniao-publica/blob/feature/primeira-tentativa-modelo/simulacao-opiniao-publica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Features:
=====================
"SEXO" => "Sexo"
"FX_ID" => "Faixa etária"
"ESCOLARIDADE" => "Escolaridade"
"P1A" => "Lembra em quem votou para Deputado Estadual nas eleições gerais de 2022?"
"P1B" => "Lembra em quem votou para Deputado Federal nas eleições gerais de 2022?"
"P1C" => "Lembra em quem votou para Senador nas eleições gerais de 2022?"
"P4" => "Nível de interesse em participar da vida política"
"RACA" => "Raça"
"RELIGIAO" => "Religião"
"REND1" => "Renda pessoal (em salários mínimos)"
"REND2" => "Renda familiar (em salários mínimos)"
"REGIAO" => "Região do país"
"COND" => "Condição do municipio"
"""

# Define the order for ordinal features
fx_id_order = [
    "16 E 17",
    "18 A 24",
    "25 A 34",
    "35 A 44",
    "45 A 54",
    "55 A 64",
    "65 E MAIS",
]
p4_order = [
    "Não sabe/ Não respondeu",
    "Nenhuma vontade",
    "Alguma vontade",
    "Muita vontade",
]
renda_order = [
    "NÃO RESPONDEU",
    "NÃO TEM RENDIMENTO PESSOAL",
    "ATÉ 1",
    "MAIS DE 1 A 2",
    "MAIS DE 2 A 5",
    "MAIS DE 5 A 10",
    "MAIS DE 10 A 20",
    "MAIS DE 20",
]

ordinal_features = {
    "FX_ID": fx_id_order,
    "P4": p4_order,
    "REND1": renda_order,
    "REND2": renda_order,
}

ordinal_feature_names = list(ordinal_features.keys())

nominal_features = [
    "SEXO",
    "ESCOLARIDADE",
    "P1A",
    "P1B",
    "P1C",
    "RACA",
    "RELIGIAO",
    "REGIAO",
    "COND",
]

selected_features = ordinal_feature_names + nominal_features

"""
*Targets*
=====================
"P2_1" => "Qual dessas propostas você acha que deveria ser prioridade de um(a) político(a)?"
"P3_1" => "Quais dessas opções você acredita que poderiam contribuir no combate à divulgação de fake news?"
"""
s_targets = ["P2_1", "P3_1"]

# =====================
# *Features ignoradas*
# # "IDADE" => "Idade"
# # "PORTE"
# # "ID_Ipec"
# # "DATA_ENTREVISTA"
# # "TIPO_COLETA"
# =====================
# *Targets ignorados*
# # "P2_2"
# # "P2_3"
# # "P3_2"
# # "P3_3"
# # "P3_4"
# # "P3_5"
# # "P3_6"
# =====================

In [ ]:
import pandas as pd

df_train = pd.read_csv(
    "https://raw.githubusercontent.com/olucasaguiar/projeto-simulacao-opiniao-publica/main/llm_simulation/data/df_train.csv"
)
df_test = pd.read_csv(
    "https://raw.githubusercontent.com/olucasaguiar/projeto-simulacao-opiniao-publica/main/llm_simulation/data/df_test.csv"
)

X_train, X_test, y_train, y_test = (
    df_train[selected_features],
    df_test[selected_features],
    df_train[s_targets],
    df_test[s_targets],
)

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

rng = np.random.RandomState(0)

## Preprocessors with Imputation
# Create a list of categories for OrdinalEncoder in the correct order
ordinal_categories = [ordinal_features[col] for col in ordinal_feature_names]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "ordinal",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=np.nan,
            ),
            ordinal_feature_names,
        ),
        (
            "one_k",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=True,
            ),
            nominal_features,
        ),
    ],
    remainder="drop",
)

In [ ]:
# @title Avaliando classificador Random Forest
import numpy as np
import scipy.stats as stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline

rng = np.random.RandomState(0)
rf_parameters = {
    "n_estimators": stats.randint(100, 500),
    "ccp_alpha": stats.loguniform(1e-5, 1e-2),
    "max_depth": stats.randint(10, 30),
    "max_features": ["sqrt", "log2", None],
    "min_samples_leaf": [1, 2, 4],
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(n_jobs=-1, class_weight="balanced", random_state=rng),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=rng),
    n_iter=20,
    random_state=rng,
    param_distributions=rf_parameters,
    n_jobs=-1,
)
rf_multiclass = MultiOutputClassifier(rf_search, n_jobs=-1)
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("estimator_search", rf_multiclass),
    ],
    verbose=True,
)

rf_pipeline.fit(X_train, y_train)
rf_y_pred = rf_pipeline.predict(X_test)

In [ ]:
rf_y_pred = rf_pipeline.predict(X_test)

In [ ]:
%pip install -q matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report


print('\n**Relatório de Métricas**')
print("\nRandom Forest:")
print(classification_report(y_test.values[:, 0], rf_y_pred[:, 0], zero_division=0))
print(classification_report(y_test.values[:, 1], rf_y_pred[:, 1], zero_division=0))

for y_col, title in enumerate(s_targets):
    disp = ConfusionMatrixDisplay.from_predictions(y_test.values[:, y_col], rf_y_pred[:, y_col], cmap='Blues')
    disp.ax_.set_title('Random Forest (CM) - %s' % title)
    disp.ax_.set_xlabel('Valores Preditos')
    disp.ax_.set_ylabel('Valores Reais')
    plt.xticks(rotation=90)
    plt.plot()